<a href="https://colab.research.google.com/github/vashirij/wildfire-tinyml-self-sufficiency/blob/main/Notebooks/06_TinyML_Model_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 TinyML Model Optimization

## Objectives

- Load the realistic wildfire dataset.
- Train several lightweight machine-learning models.
- Compare accuracy, precision, recall, F1-score, inference time, training time, and model size.
- Select the best TinyML candidate.
- Save models, metrics, and deployment artifacts.


In [ ]:
from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT=Path('/content/drive/MyDrive/WildfireProject')
DATA_PATH=PROJECT_ROOT/'data'/'simulated'/'realistic'/'realistic_environment_stream.csv'
MODEL_DIR=PROJECT_ROOT/'models'/'tinyml'
RESULT_DIR=PROJECT_ROOT/'results'/'tinyml'
FIG_DIR=PROJECT_ROOT/'figures'/'tinyml'
for d in [MODEL_DIR,RESULT_DIR,FIG_DIR]:
    d.mkdir(parents=True,exist_ok=True)

df=pd.read_csv(DATA_PATH)

FEATURES=['temperature_c','humidity_percent','smoke_ppm','co_ppm','wind_speed_kmh']
X=df[FEATURES]
y=df['wildfire']

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42
)

models={
'Decision Tree':DecisionTreeClassifier(max_depth=8,random_state=42),
'Random Forest':RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,n_jobs=-1),
'Extra Trees':ExtraTreesClassifier(n_estimators=100,max_depth=10,random_state=42,n_jobs=-1),
'Gradient Boosting':GradientBoostingClassifier(random_state=42),
'Logistic Regression':LogisticRegression(max_iter=1000),
'Gaussian NB':GaussianNB()
}

rows=[]
best_model=None
best_f1=-1

for name,model in models.items():
    t0=time.perf_counter()
    model.fit(X_train,y_train)
    train_time=time.perf_counter()-t0

    t0=time.perf_counter()
    pred=model.predict(X_test)
    infer_time=(time.perf_counter()-t0)/len(X_test)

    path=MODEL_DIR/(name.replace(' ','_')+'.joblib')
    joblib.dump(model,path)
    size_kb=path.stat().st_size/1024

    row={
        'model':name,
        'accuracy':accuracy_score(y_test,pred),
        'precision':precision_score(y_test,pred,zero_division=0),
        'recall':recall_score(y_test,pred,zero_division=0),
        'f1_score':f1_score(y_test,pred,zero_division=0),
        'training_time_s':train_time,
        'inference_time_ms':infer_time*1000,
        'model_size_kb':size_kb
    }
    rows.append(row)

    if row['f1_score']>best_f1:
        best_f1=row['f1_score']
        best_model=name

results=pd.DataFrame(rows).sort_values('f1_score',ascending=False)
display(results.round(4))

results.to_csv(RESULT_DIR/'week6_model_metrics.csv',index=False)

best_path=MODEL_DIR/(best_model.replace(' ','_')+'.joblib')
joblib.dump(joblib.load(best_path),MODEL_DIR/'tinyml_best_model.joblib')

plt.figure(figsize=(8,4))
plt.bar(results['model'],results['f1_score'])
plt.xticks(rotation=20,ha='right')
plt.ylabel('F1-score')
plt.tight_layout()
plt.savefig(FIG_DIR/'model_accuracy_comparison.png',dpi=300)
plt.show()

plt.figure(figsize=(8,4))
plt.bar(results['model'],results['inference_time_ms'])
plt.xticks(rotation=20,ha='right')
plt.ylabel('Inference Time (ms/sample)')
plt.tight_layout()
plt.savefig(FIG_DIR/'inference_time_comparison.png',dpi=300)
plt.show()

plt.figure(figsize=(8,4))
plt.bar(results['model'],results['model_size_kb'])
plt.xticks(rotation=20,ha='right')
plt.ylabel('Model Size (KB)')
plt.tight_layout()
plt.savefig(FIG_DIR/'model_size_comparison.png',dpi=300)
plt.show()

plt.figure(figsize=(6,5))
plt.scatter(results['model_size_kb'],results['f1_score'])
for _,r in results.iterrows():
    plt.text(r['model_size_kb'],r['f1_score'],r['model'],fontsize=8)
plt.xlabel('Model Size (KB)')
plt.ylabel('F1-score')
plt.tight_layout()
plt.savefig(FIG_DIR/'pareto_front.png',dpi=300)
plt.show()

summary=results.iloc[0]
print('='*60)
print('TINYML OPTIMIZATION COMPLETE')
print('='*60)
print('Best model:',summary['model'])
print('F1-score:',round(summary['f1_score'],4))
print('Model size (KB):',round(summary['model_size_kb'],2))
print('Inference (ms):',round(summary['inference_time_ms'],4))
